# Projeto Olist: Análise Exploratória com SQL e Pandas

Este notebook faz parte de um estudo de caso focado em **Engenharia e Análise de Dados**. O objetivo é extrair insights de negócio a partir de um banco de dados relacional (SQLite) utilizando a biblioteca Pandas para manipulação e cálculos estatísticos.

## 1. Configuração do Ambiente e Conexão
Nesta etapa inicial, importamos as bibliotecas necessárias e estabelecemos a conexão com o banco de dados `olist.db`, gerado previamente via script de automação.

In [1]:
import pandas as pd

In [2]:
import sqlite3

# Conectar ao banco (lembre-se: como o notebook está na pasta 'notebooks', 
# precisamos usar '../' para voltar uma pasta e achar a 'data')
conn = sqlite3.connect('../data/olist.db')

# Ler os primeiros 5 clientes para testar
df = pd.read_sql_query("SELECT * FROM customers LIMIT 5", conn)
df

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## 3. Distribuição Geográfica de Clientes
Utilizamos funções de agregação em SQL (`COUNT` e `GROUP BY`) para identificar os principais polos de consumo da plataforma. Esta análise ajuda a entender a concentração de mercado por cidade.

In [3]:
query = """
SELECT 
    customer_city, 
    COUNT(customer_id) as total_clientes
FROM customers
GROUP BY customer_city
ORDER BY total_clientes DESC
LIMIT 5
"""

df_top_cidades = pd.read_sql_query(query, conn)
df_top_cidades

,customer_city,total_clientes
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521


Observamos que a cidade de São Paulo representa o maior volume de clientes, indicando uma alta concentração logística e de mercado na região Sudeste.

## 4. Desempenho Financeiro por Categoria
Para responder perguntas de negócio sobre receita, realizamos um **JOIN** entre as tabelas de itens de pedido e produtos. O objetivo é ranquear as categorias que mais contribuem para o faturamento total da Olist.

In [4]:
query = """
SELECT 
    t2.product_category_name,
    SUM(t1.price) as faturamento_total
FROM order_items as t1
LEFT JOIN products as t2
    ON t1.product_id = t2.product_id
GROUP BY t2.product_category_name
ORDER BY faturamento_total DESC
LIMIT 10
"""

df_faturamento = pd.read_sql_query(query, conn)
df_faturamento

,product_category_name,faturamento_total
0,beleza_saude,1258681.34
1,relogios_presentes,1205005.68
2,cama_mesa_banho,1036988.68
3,esporte_lazer,988048.97
4,informatica_acessorios,911954.32
5,moveis_decoracao,729762.49
6,cool_stuff,635290.85
7,utilidades_domesticas,632248.66
8,automotivo,592720.11
9,ferramentas_jardim,485256.46


## 5. Faturamento por Localização do Vendedor
Nesta análise, tratamos os Estados de origem dos vendedores como "filiais" para entender a distribuição geográfica da receita. Utilizamos um `JOIN` entre as tabelas de itens e vendedores para consolidar os valores financeiros por unidade federativa.

In [5]:
query_filiais = """
SELECT 
    t2.seller_state as estado_vendedor,
    SUM(t1.price) as faturamento_total
FROM order_items as t1
JOIN sellers as t2 ON t1.seller_id = t2.seller_id
GROUP BY t2.seller_state
ORDER BY faturamento_total DESC
"""
df_filiais = pd.read_sql_query(query_filiais, conn)
df_filiais

,estado_vendedor,faturamento_total
0,SP,8753396.21
1,PR,1261887.21
2,MG,1011564.74
3,RJ,843984.22
4,SC,632426.07
5,RS,378559.54
6,BA,285561.56
7,DF,97749.48
8,PE,91493.85
9,GO,66399.21


## 6. Participação de Mercado (Share) por Categoria
Para identificar a relevância de cada departamento no faturamento global, calculamos o percentual de participação (Share). Aqui, combinamos o poder de agregação do **SQL** para a extração dos dados com a flexibilidade do **Pandas** para o cálculo das métricas de proporção.

In [6]:
# Primeiro pegamos o faturamento por categoria
query_cat = """
SELECT 
    t2.product_category_name,
    SUM(t1.price) as faturamento
FROM order_items as t1
JOIN products as t2 ON t1.product_id = t2.product_id
GROUP BY t2.product_category_name
"""
df_participacao = pd.read_sql_query(query_cat, conn)

# Calculando o percentual no Pandas
total_geral = df_participacao['faturamento'].sum()
df_participacao['percentual'] = (df_participacao['faturamento'] / total_geral) * 100

df_participacao.sort_values('percentual', ascending=False).head(10)

,product_category_name,faturamento,percentual
12,beleza_saude,1258681.34,9.260700
67,relogios_presentes,1205005.68,8.865783
14,cama_mesa_banho,1036988.68,7.629605
33,esporte_lazer,988048.97,7.269533
45,informatica_acessorios,911954.32,6.709669
55,moveis_decoracao,729762.49,5.369200
27,cool_stuff,635290.85,4.674128
73,utilidades_domesticas,632248.66,4.651745
9,automotivo,592720.11,4.360916
41,ferramentas_jardim,485256.46,3.570256


## 7. Análise Cruzada: Categoria vs. Região (Pivot Table)
Para entender o comportamento de consumo regional, criamos uma Tabela Dinâmica (*Pivot Table*) cruzando as categorias de produtos com os estados dos clientes. Essa técnica permite visualizar padrões de preferência geográfica de forma multidimensional.

In [7]:
# Query para trazer os dados necessários
query_pivot = """
SELECT 
    t3.customer_state,
    t2.product_category_name,
    t1.price
FROM order_items as t1
JOIN products as t2 ON t1.product_id = t2.product_id
JOIN orders as o ON t1.order_id = o.order_id
JOIN customers as t3 ON o.customer_id = t3.customer_id
WHERE t3.customer_state IN ('SP', 'RJ', 'MG', 'RS', 'PR') -- Filtrando os 5 maiores estados
"""

df_raw = pd.read_sql_query(query_pivot, conn)

# Criando a Pivot Table no Pandas
pivot_result = df_raw.pivot_table(
    index='product_category_name', 
    columns='customer_state', 
    values='price', 
    aggfunc='sum'
).fillna(0)

# Mostrando as top 10 categorias
pivot_result.sort_values('SP', ascending=False).head(10)

customer_state,MG,PR,RJ,RS,SP
product_category_name,,,,,
cama_mesa_banho,129643.98,45598.47,148035.23,60270.74,478284.52
beleza_saude,157558.30,54949.36,145298.62,51327.72,462305.22
relogios_presentes,123759.23,59967.04,185379.65,48152.53,435009.92
esporte_lazer,112719.19,58630.20,125148.04,53788.32,386357.01
informatica_acessorios,111069.74,44127.09,120289.45,52757.73,350747.88
moveis_decoracao,78999.74,49484.89,98427.86,54813.58,286708.02
utilidades_domesticas,78073.72,31221.60,77675.57,38994.63,275378.63
automotivo,72851.08,28134.02,65838.10,26333.12,214277.27
cool_stuff,72778.48,37539.12,83909.95,46676.34,213186.89


## 7. Técnicas de Seleção e Fatiamento
Exploramos o uso do método `.iloc`, que permite a seleção de dados baseada em sua posição (índices). Esta técnica é útil para extrair fatias específicas do DataFrame (*slicing*), independentemente dos rótulos das colunas.

In [8]:
# Pegando as 10 primeiras linhas e as 3 primeiras colunas
df_participacao.iloc[:10, 0:3]

,product_category_name,faturamento,percentual
0,NaN,179535.28,1.320924
1,agro_industria_e_comercio,72530.47,0.533640
2,alimentos,29393.41,0.216261
3,alimentos_bebidas,15179.48,0.111682
4,artes,24202.64,0.178070
5,artes_e_artesanato,1814.01,0.013347
6,artigos_de_festas,4485.18,0.033000
7,artigos_de_natal,8800.82,0.064752
8,audio,50688.50,0.372939
9,automotivo,592720.11,4.360916


## 8. Filtros Condicionais
Nesta etapa, aplicamos filtros lógicos para isolar registros que atendem a critérios específicos de negócio. Utilizamos operadores booleanos (`&` para 'e', `|` para 'ou') para construir consultas complexas diretamente no DataFrame, simulando a cláusula `WHERE` do SQL, porém com a performance em memória do Pandas.

In [9]:
# Criando um filtro condicional complexo
filtro_elite = (df_participacao['faturamento'] > 100000) & (df_participacao['percentual'] > 5)

# Aplicando o filtro
categorias_premium = df_participacao[filtro_elite]
categorias_premium

,product_category_name,faturamento,percentual
12,beleza_saude,1258681.34,9.260700
14,cama_mesa_banho,1036988.68,7.629605
33,esporte_lazer,988048.97,7.269533
45,informatica_acessorios,911954.32,6.709669
55,moveis_decoracao,729762.49,5.369200
67,relogios_presentes,1205005.68,8.865783


No Pandas, o Índice não é apenas um contador de linhas; ele é o "endereço" do dado

.iloc: Você usa números (posições). É como dizer: "Vá na gaveta 1 e pegue os itens de 0 a 5".

.loc ou Filtro Direto: Você usa rótulos ou condições. É como dizer: "Pegue todos os itens que custam mais de 10 reais".

## 9. Operações com Índices
A manipulação de índices é essencial para a organização e performance das consultas no Pandas. Nesta seção, exploramos como definir uma coluna como identificador principal (`set_index`), como retornar ao estado original (`reset_index`) e como renomear ou acessar metadados do DataFrame através de `.index` e `.columns`.

Útil para preparar dados para Gráficos (o Matplotlib e o Seaborn usam o índice para rotular os eixos).

In [12]:
# Definindo a categoria como índice
df_cat_index = df_participacao.set_index('product_category_name')

# Ver os dados apenas de 'perfumaria'
print(df_cat_index.loc['perfumaria'])



faturamento    399124.870000
percentual          2.936546
Name: perfumaria, dtype: float64


In [14]:
# Resetando o índice (útil quando você quer transformar o índice de volta em coluna)
df_original = df_cat_index.reset_index()

# Acessando os nomes das colunas e o tipo do índice
print(df_participacao.columns)
print(df_participacao.index)


Index(['product_category_name', 'faturamento', 'percentual'], dtype='str')
RangeIndex(start=0, stop=74, step=1)


## 10. Índices Multiníveis (MultiIndex)
A criação de índices hierárquicos permite trabalhar com dimensões superiores em uma estrutura bidimensional (DataFrame). Esta técnica é fundamental para organizar dados que possuem uma subordinação natural, como divisões geográficas (Estado > Cidade) ou temporais (Ano > Mês). Utilizamos o método `pd.MultiIndex.from_tuples()` para estruturar essa hierarquia, facilitando agregações complexas e seleções multidimensionais.

No dia a dia, muitas vezes o MultiIndex surge "sozinho" quando fazemos um groupby com duas colunas.

Para consolidar o aprendizado de índices multiníveis, realizamos uma análise de faturamento cruzando a localização do cliente (`customer_state`) com a categoria do produto (`product_category_name`). Ao agrupar os dados por múltiplas colunas, o Pandas gera automaticamente um **MultiIndex**, permitindo uma navegação hierárquica pelos resultados de venda da Olist.

In [16]:
# Query para buscar Estado, Categoria e Preço
query_multi = """
SELECT 
    t3.customer_state,
    t2.product_category_name,
    t1.price
FROM order_items as t1
JOIN products as t2 ON t1.product_id = t2.product_id
JOIN orders as o ON t1.order_id = o.order_id
JOIN customers as t3 ON o.customer_id = t3.customer_id
"""

df_vendas_geo = pd.read_sql_query(query_multi, conn)

# Criando o agrupamento que gera o MultiIndex automaticamente.  Agrupamos por Estado e Categoria, somando o Preço
analise_multi = df_vendas_geo.groupby(['customer_state', 'product_category_name']).sum()

analise_multi

price
customer_state product_category_name               
AC             artigos_de_natal               69.90
               automotivo                    540.98
               bebes                         697.84
               beleza_saude                 1386.58
               brinquedos                    234.79
...                                             ...
TO             portateis_casa_forno_e_cafe  1999.00
               relogios_presentes           5446.89
               telefonia                    1268.12
               telefonia_fixa               1390.99
               utilidades_domesticas        1275.75

[1368 rows x 1 columns]

In [ ]:
# método Cross-section: use o .xs para responder perguntas como: "Como a categoria X performa em diferentes regiões?"
analise_multi.xs('perfumaria', level='product_category_name')

,price
customer_state,
AC,303.88
AL,1766.66
AM,94.89
BA,9723.01
CE,8747.73
DF,9948.04
ES,6577.55
GO,10291.58
MA,4436.58


## 11. Tratamento de Dados Ausentes (Missing Data)
A presença de valores nulos (`NaN`) é comum em datasets reais e pode comprometer análises estatísticas. Exploramos estratégias de limpeza e imputação, como a remoção seletiva (`dropna`), o preenchimento com valores constantes ou estatísticos (`fillna`) e métodos de propagação temporal (`ffill` e `bfill`). A escolha da estratégia depende do impacto do dado ausente no contexto de negócio.

In [32]:
# Identificando onde estão os nulos
query_produtos = "SELECT * FROM products"
df_produtos = pd.read_sql_query(query_produtos, conn)

# Para uma tomada de decisão assertiva sobre a estratégia de limpeza, calculamos a representatividade dos 
# valores ausentes em cada coluna. Esta métrica é vital para identificar se a perda de dados é estatisticamente
# significativa em relação ao volume total do dataset de produtos.

print("Valores nulos por coluna:")

# Calculando a soma de nulos e o total de registros
nulos_contagem = df_produtos.isnull().sum()
total_registros = len(df_produtos)

# Criando um novo DataFrame para consolidar as informações
df_diagnostico = pd.DataFrame({
    'Total_Nulos': nulos_contagem,
    'Percentual_%': (nulos_contagem / total_registros) * 100
})

# Filtrando apenas onde há nulos e ordenando
df_diagnostico = df_diagnostico[df_diagnostico['Total_Nulos'] > 0].sort_values('Percentual_%', ascending=False)

df_diagnostico



Valores nulos por coluna:


,Total_Nulos,Percentual_%
product_category_name,610,1.851234
product_name_lenght,610,1.851234
product_description_lenght,610,1.851234
product_photos_qty,610,1.851234
product_weight_g,2,0.006070
product_length_cm,2,0.006070
product_height_cm,2,0.006070
product_width_cm,2,0.006070


A ausência de categorias em menos de 2% da base permite o preenchimento por um valor genérico ('Outros') sem distorcer o comportamento das categorias principais.

In [28]:
# Exemplo de fillna: Preenchendo categorias vazias com 'nao_informado'
df_produtos['product_category_name'] = df_produtos['product_category_name'].fillna('nao_informado')
df_produtos['product_category_name'].isna().sum()



np.int64(0)

### 11.1. Refinando a Limpeza: Imputação em vez de Exclusão
Após uma análise crítica, optamos por não excluir registros com valores nulos nas características físicas dos produtos (fotos, peso e dimensões). A exclusão poderia causar inconsistências em análises de faturamento futuro. Em vez disso, aplicamos a **imputação de dados**:
- **Fotos:** Valores ausentes preenchidos com `0`.
- **Dimensões e Peso:** Preenchidos com a **mediana**, garantindo que o produto ainda exista para cálculos de receita, mesmo com dados técnicos incompletos.

Excluir é sempre a última opção, deve-se manter a integridade referencial.

In [ ]:
# Tratando a quantidade de fotos (preenchendo com 0)
df_produtos['product_photos_qty'] = df_produtos['product_photos_qty'].fillna(0)

# Tratando peso e dimensões com a MEDIANA
# Usamos a mediana porque se houver um produto extremamente pesado (outlier), 
# ele não puxa o valor para cima como a média faria.
df_produtos['product_weight_g'] = df_produtos['product_weight_g'].fillna(df_produtos['product_weight_g'].median())
df_produtos['product_length_cm'] = df_produtos['product_length_cm'].fillna(df_produtos['product_length_cm'].median())
df_produtos['product_height_cm'] = df_produtos['product_height_cm'].fillna(df_produtos['product_height_cm'].median())
df_produtos['product_width_cm'] = df_produtos['product_width_cm'].fillna(df_produtos['product_width_cm'].median())

# Verificando se ainda restam nulos (exceto na categoria, que já tratamos como 'nao_informado')
print("Nulos restantes:")
print(df_produtos.isnull().sum())

np.int64(0)